# 1. Data Collection: SpaceX API and Web Scraping
**Goal:** collect launch records plus reference information from the public SpaceX API. Then inspect the course-relevant launch history page for a web-scraping exercise.

The API is a live source; the exact counts can change. This notebook saves raw JSON and flattened CSV files, plus a manifest documenting the retrieval time.

In [13]:

# ============================================================
# SPACEX CAPSTONE — COMPLETE DATA COLLECTION (SINGLE CELL)
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import time
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

# 1. Project folders
PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 2. API configuration
API_BASES = [
    "https://api.spacexdata.com/v4",
    "https://gateway.pipeworx.io/spacex/v4",
]
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; SpaceXDataScienceCourse/1.0)"
}
ENDPOINTS = ["launches", "rockets", "launchpads", "payloads", "cores"]
REQUIRED_ENDPOINTS = {"launches", "rockets"}

session = requests.Session()
session.headers.update(HEADERS)


# 3. Normalize API responses
def normalize_api_payload(payload, url):
    if isinstance(payload, list):
        records = payload
    elif isinstance(payload, dict):
        records = next(
            (
                payload[key]
                for key in ("docs", "data", "results", "items")
                if isinstance(payload.get(key), list)
            ),
            None,
        )
        if records is None:
            raise ValueError(
                f"Unexpected API response from {url}: {list(payload.keys())[:10]}"
            )
    else:
        raise ValueError(f"Unexpected response type from {url}")

    if not records:
        raise ValueError(f"Empty dataset returned by {url}")

    return records


# 4. Fetch with retries and API fallback
def fetch_json(endpoint, retries_per_base=2):
    errors = []

    for base in API_BASES:
        url = f"{base.rstrip('/')}/{endpoint}"

        for attempt in range(1, retries_per_base + 1):
            try:
                response = session.get(url, timeout=30)
                response.raise_for_status()
                records = normalize_api_payload(response.json(), url)

                print(
                    f"SUCCESS: {endpoint} — {len(records)} records "
                    f"from {base}"
                )
                return records, url

            except (requests.RequestException, ValueError) as exc:
                message = (
                    f"{url} | attempt {attempt}/{retries_per_base}: {exc}"
                )
                errors.append(message)
                print("WARNING:", message)

                if attempt < retries_per_base:
                    time.sleep(attempt * 2)

    raise RuntimeError(
        f"Unable to download '{endpoint}'.\n" + "\n".join(errors)
    )


# 5. Download and save datasets
records_by_endpoint = {}
manifest = {
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "datasets": {},
    "errors": [],
}

for endpoint in ENDPOINTS:
    try:
        records, source_url = fetch_json(endpoint)
    except Exception as exc:
        message = f"{endpoint}: {exc}"
        manifest["errors"].append(message)
        print("\nDATASET UNAVAILABLE:", message)

        if endpoint in REQUIRED_ENDPOINTS:
            raise RuntimeError(
                f"Required dataset '{endpoint}' is unavailable. "
                "Data collection cannot continue until a working source "
                "is available."
            ) from exc

        # Optional datasets may be missing, but are marked unavailable.
        continue

    records_by_endpoint[endpoint] = records
    frame = pd.json_normalize(records)

    json_file = f"spacex_{endpoint}.json"
    csv_file = f"spacex_{endpoint}.csv"

    (DATA_DIR / json_file).write_text(
        json.dumps(records, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    frame.to_csv(DATA_DIR / csv_file, index=False)

    manifest["datasets"][endpoint] = {
        "source_url": source_url,
        "record_count": len(frame),
        "column_count": len(frame.columns),
        "columns": list(frame.columns),
        "json_file": json_file,
        "csv_file": csv_file,
    }

    print(f"SAVED: {json_file} and {csv_file}")
    time.sleep(0.25)


# 6. Save collection report, even if optional endpoints failed
(DATA_DIR / "collection_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\n" + "=" * 55)
print("DATA COLLECTION SUMMARY")
print("=" * 55)
print("Project folder:", PROJECT_DIR.resolve())
print("Data folder:", DATA_DIR.resolve())
print("Available datasets:")

for name, records in records_by_endpoint.items():
    print(f"  {name}: {len(records)} records")

print("Unavailable datasets:", manifest["errors"])
print("Manifest saved:", (DATA_DIR / "collection_manifest.json").exists())

# 7. Preview launch data
if "launches" in records_by_endpoint:
    launches_raw = pd.json_normalize(records_by_endpoint["launches"])
    print("\nLaunches preview:")
    display(launches_raw.head(3))
    print("Launches shape:", launches_raw.shape)
    print("Columns:", launches_raw.columns.tolist())

# 8. Optional web scraping check
SCRAPE_URL = (
    "https://en.wikipedia.org/wiki/"
    "List_of_Falcon_9_and_Falcon_Heavy_launches"
)

try:
    response = session.get(SCRAPE_URL, timeout=20)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    tables = pd.read_html(str(soup))

    print("\nWeb scraping successful.")
    print("Tables found:", len(tables))

    if tables:
        tables[0].to_csv(
            DATA_DIR / "web_scrape_table0_inspection.csv",
            index=False,
        )
        print("Saved an inspection table.")

except Exception as exc:
    print("\nWeb scraping unavailable:", exc)
    print("API data collection is separate from this optional step.")

print("\nCell execution finished.")


SUCCESS: launches — 101 records from https://gateway.pipeworx.io/spacex/v4
SAVED: spacex_launches.json and spacex_launches.csv
SUCCESS: rockets — 13 records from https://gateway.pipeworx.io/spacex/v4
SAVED: spacex_rockets.json and spacex_rockets.csv

DATASET UNAVAILABLE: launchpads: Unable to download 'launchpads'.
https://api.spacexdata.com/v4/launchpads | attempt 1/2: 525 Server Error: <none> for url: https://api.spacexdata.com/v4/launchpads
https://api.spacexdata.com/v4/launchpads | attempt 2/2: 525 Server Error: <none> for url: https://api.spacexdata.com/v4/launchpads
https://gateway.pipeworx.io/spacex/v4/launchpads | attempt 1/2: 404 Client Error: Not Found for url: https://gateway.pipeworx.io/spacex/v4/launchpads
https://gateway.pipeworx.io/spacex/v4/launchpads | attempt 2/2: 404 Client Error: Not Found for url: https://gateway.pipeworx.io/spacex/v4/launchpads

DATASET UNAVAILABLE: payloads: Unable to download 'payloads'.
https://api.spacexdata.com/v4/payloads | attempt 1/2: 525 

,name,status,success,date_utc,pad,rocket,details,links.article,links.webcast,links.wikipedia
0,Falcon Heavy | NROL-97,Launch Successful,True,2026-10-02T03:54:00Z,NaN,NaN,NaN,NaN,NaN,NaN
1,Falcon 9 Block 5 | Transporter 18 (Dedicated S...,Launch Successful,True,2026-10-01T18:32:00Z,NaN,NaN,NaN,NaN,NaN,NaN
2,Falcon 9 Block 5 | Crew-13,Launch Successful,True,2026-10-01T15:10:06Z,NaN,NaN,NaN,NaN,NaN,NaN


Launches shape: (101, 10)
Columns: ['name', 'status', 'success', 'date_utc', 'pad', 'rocket', 'details', 'links.article', 'links.webcast', 'links.wikipedia']


/tmp/ipykernel_722/4150148914.py:186: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))



Web scraping successful.
Tables found: 19
Saved an inspection table.

Cell execution finished.


In [14]:
if "records_by_endpoint" not in globals() or "launches" not in records_by_endpoint:
    raise RuntimeError("Run the data collection cell successfully before previewing launches.")

launches_raw = pd.json_normalize(records_by_endpoint["launches"])
display(launches_raw.head(5))
print("Launches shape:", launches_raw.shape)
print("Column names:")
print(launches_raw.columns.tolist())

,name,status,success,date_utc,pad,rocket,details,links.article,links.webcast,links.wikipedia
0,Falcon Heavy | NROL-97,Launch Successful,True,2026-10-02T03:54:00Z,NaN,NaN,NaN,NaN,NaN,NaN
1,Falcon 9 Block 5 | Transporter 18 (Dedicated S...,Launch Successful,True,2026-10-01T18:32:00Z,NaN,NaN,NaN,NaN,NaN,NaN
2,Falcon 9 Block 5 | Crew-13,Launch Successful,True,2026-10-01T15:10:06Z,NaN,NaN,NaN,NaN,NaN,NaN
3,Starship | Starlink Group 31-1 (Starship Fligh...,Launch Successful,True,2026-09-28T12:48:59Z,NaN,NaN,NaN,NaN,NaN,NaN
4,Falcon 9 Block 5 | USSF-385,Launch Successful,True,2026-09-26T14:00:54Z,NaN,NaN,NaN,NaN,NaN,NaN


Launches shape: (101, 10)
Column names:
['name', 'status', 'success', 'date_utc', 'pad', 'rocket', 'details', 'links.article', 'links.webcast', 'links.wikipedia']


## Web scraping methodology
The following cell fetches a publicly accessible launch-history page and inspects its tables. Confirm the page's terms and robots guidance, and use the course lab's specified URL/selectors if they differ. HTML structures can change, so validate table headers before relying on scraped records.

In [15]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
scrape_status = None
scraped_table_headers = []
try:
    r = requests.get(SCRAPE_URL, headers=HEADERS, timeout=30)
    scrape_status = r.status_code
    print("Page status:", scrape_status, "| Content type:", r.headers.get("content-type"))
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    tables = soup.find_all("table")
    print("Tables found:", len(tables))
    for i, table in enumerate(tables[:5]):
        headers = [th.get_text(" ", strip=True) for th in table.find_all("th")]
        scraped_table_headers.append(headers)
        print(f"Table {i} headers:", headers[:15])
except requests.RequestException as e:
    print("Web scraping request failed:", repr(e))
    print("Continue with API data; retry scraping later or use the course-provided page.")

Page status: 200 | Content type: text/html; charset=UTF-8
Tables found: 19
Table 0 headers: ['List of Falcon 9 and Falcon Heavy launches']
Table 1 headers: ['Flight No.', 'Date and time ( UTC )', 'Version, booster [ i ]', 'Launch site', 'Payload [ j ]', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing', '418', '419', '420', '421', '422']
Table 2 headers: ['Flight No.', 'Date and time ( UTC )', 'Version, booster [ i ]', 'Launch site', 'Payload [ j ]', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing', '583', '584', '585', '586', '587']
Table 3 headers: ['Date and time ( UTC )', 'Version, booster [ i ]', 'Launch site', 'Payload [ j ]', 'Orbit', 'Customer']
Table 4 headers: ['Date and time ( UTC )', 'Version, booster [ i ]', 'Launch site', 'Payload [ j ]', 'Orbit', 'Customer']


In [16]:
# Optional: parse the first table only after inspecting its headers.
# This does not assume that the first table is necessarily the desired launch table.
if 'soup' in globals() and soup.find_all("table"):
    try:
        html_tables = pd.read_html(str(soup))
        print("Pandas parsed tables:", len(html_tables))
        if html_tables:
            display(html_tables[0].head())
            html_tables[0].to_csv(DATA_DIR / "web_scrape_table0_inspection.csv", index=False)
            print("Saved inspection copy: web_scrape_table0_inspection.csv")
    except Exception as e:
        print("Table parsing needs adjustment after inspecting the current page:", repr(e))

/tmp/ipykernel_722/3482206065.py:5: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  html_tables = pd.read_html(str(soup))


Pandas parsed tables: 19


,List of Falcon 9 and Falcon Heavy launches
0,2010–2019 · 2020–2022 · 2023 · 2024 · 2025 · 2...


Saved inspection copy: web_scrape_table0_inspection.csv


## Deliverables and interpretation
- Raw API JSON and flattened CSVs for launches, rockets, launchpads, payloads and cores.
- `collection_manifest.json` with collection time, record counts and columns.
- A web-scraping inspection file if the page/table parser succeeds.
- In the presentation, describe the endpoint calls, the inspected webpage, and any errors honestly.